In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

## Load data

In [2]:
df_train = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_train.csv")
df_test = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_test.csv")

In [3]:
columns_remove = [
    'VitaminD',
    'YearStart',
]

In [4]:
df_train = df_train[df_train['milk_consumption']<=3]
df_test = df_test[df_test['milk_consumption']<=3]

In [5]:
df_train.drop(columns=columns_remove, inplace=True)
df_test.drop(columns=columns_remove, inplace=True)

In [6]:
category_columns = [
    'Gender','Race' ,'label','milk_consumption'
]

In [7]:
df_train.columns

Index(['Gender', 'Age', 'Race', 'familysize', 'PIR', 'BMI',
       'WaistCircumference', 'FastingGlucose', 'ALT', 'AST',
       'AlkalinePhosphotase', 'Triglycerides', 'UricAcid', 'Creatinine',
       'HDLCholesterol', 'LDLCholesterol', 'Hemoglobin', 'Hematocrit',
       'MeanCellVolumn', 'MeanCellHemoglobin', 'RedCellDistributionWidth',
       'PlateletCount', 'MeanPlateletVolume', 'Hba1c', 'SmokeFam',
       'milk_consumption', 'label'],
      dtype='object')

In [7]:
unuseful_features = ['SmokeFam','WaistCircumference','AST','ALT','AlkalinePhosphotase','UricAcid','LDLCholesterol','Hematocrit','MeanCellHemoglobin','PlateletCount', 'MeanPlateletVolume','familysize']

In [8]:
df_train.drop(columns=unuseful_features,inplace=True)
df_test = df_test[df_train.columns]

## Training:  SMOTE with SVMSMOTE

In [9]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# ========== 0) Assume dataset split ==========
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col != 'label']

# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)
dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}

# ====== 2) Base classifiers for stacking ======
# base_learners = [
#     ('lightgbm', LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
#     ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
#     ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42)),
# ]
from sklearn.ensemble import GradientBoostingClassifier

gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}
# 🔑 Notes on each setup:

# Balanced → vanilla RF, safe baseline.

# ShallowFast → small forest, faster training, may underfit.

# Deep → more complex trees, higher variance, may capture interactions.

# Regularized → controls overfitting with splits/leaves.

# Conservative → very large forest, but heavily regularized.

# RobustSubsample → adds randomness via subsampling + limited features.

# Lightweight → good for quick prototyping on large datasets.

# Heavy → huge ensemble, slow but highly stable (overkill for small datasets).

# 👉 Would you like me to also design a meta-list of “best RF defaults” (like your base_learners) so you can plug them into your stacking/blending pipeline directly?
rf_setups = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.6, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}

#Best: GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)
# base_learners = [
#     ('lightgbm', LGBMClassifier(
#         n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
#     # ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
#     #('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)),
#     ('RandomForest', rf_setups['Regularized']),
#     ('dt', dt_setups['Shallow']),  
#     # ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
#     ('SVM', SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)),
#     ('nb', GaussianNB(var_smoothing= 1e-10)),
# ]
base_learners = [
    ('lightgbm', LGBMClassifier(
        n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    # ('xgboost', XGBClassifier(
    #     n_estimators=100, max_depth=6, learning_rate=0.06, subsample=0.8, colsample_bytree=0.8,
    #     random_state=42, eval_metric="logloss", use_label_encoder=False, verbosity=0)),
    ('rf', rf_setups['Regularized']),   # strong tree bagging
    ('nb', GaussianNB(var_smoothing= 1e-10)),
    # ('svm', SVC(kernel='rbf', C=10, gamma=0.01, probability=True, class_weight='balanced', random_state=42)),
    # ('lr', LogisticRegression(
    #     max_iter=2000, solver='saga', penalty='elasticnet',
    #     l1_ratio=0.5, C=0.5, class_weight='balanced', random_state=42))
]

# ====== 3) Meta-learner ======
# meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    penalty='elasticnet',   # mix between L1 and L2
    l1_ratio=0.6,           # 0.0 -> pure L2, 1.0 -> pure L1
    C=1.0,                  # regularization strength
    class_weight='balanced',
    random_state=42
)
# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  # use probabilities from base models
    n_jobs=-1
)

# ====== 5) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='minority',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 6) Full pipeline with SVMSMOTE + Stacking ======
stacking_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote1', svmsmote),
    ('classifier', stacking_clf)
])

# ====== 7) Train and evaluate ======
print("\n🚀 Training Ensemble Stacking Model with SVMSMOTE...")
stacking_pipeline.fit(X_train_raw, y_train)

y_pred = stacking_pipeline.predict(X_test_raw)
y_proba = stacking_pipeline.predict_proba(X_test_raw)

# Evaluate
if len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba[:, 1])
else:
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
accuracy = accuracy_score(y_test, y_pred)
p_class, r_class, f1_class, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)
print(f1_w)
print("\n📊 Per-class metrics:")
for i in range(len(p_class)):
    print(f"Class {i}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")

print("\n✅ Ensemble Stacking Results")
print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")



🚀 Training Ensemble Stacking Model with SVMSMOTE...
0.7133400303013966

📊 Per-class metrics:
Class 0: P=0.8034, R=0.7667, F1=0.7846, Support=3193
Class 1: P=0.5294, R=0.5832, F1=0.5550, Support=1437

✅ Ensemble Stacking Results
Accuracy: 0.7097, F1: 0.6698, AUC: 0.7361


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [11]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# ========== 0) Assume dataset split ==========
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col != 'label']

# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)
dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}

# ====== 2) Base classifiers for stacking ======
# base_learners = [
#     ('lightgbm', LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
#     ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
#     ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42)),
# ]
from sklearn.ensemble import GradientBoostingClassifier

gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}
# 🔑 Notes on each setup:

# Balanced → vanilla RF, safe baseline.

# ShallowFast → small forest, faster training, may underfit.

# Deep → more complex trees, higher variance, may capture interactions.

# Regularized → controls overfitting with splits/leaves.

# Conservative → very large forest, but heavily regularized.

# RobustSubsample → adds randomness via subsampling + limited features.

# Lightweight → good for quick prototyping on large datasets.

# Heavy → huge ensemble, slow but highly stable (overkill for small datasets).

# 👉 Would you like me to also design a meta-list of “best RF defaults” (like your base_learners) so you can plug them into your stacking/blending pipeline directly?
rf_setups = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.6, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}

#Best: GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)
base_learners = [
    ('lightgbm', LGBMClassifier(
        n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    # ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
    #('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)),
    ('RandomForest', rf_setups['Regularized']),
    ('dt', dt_setups['Shallow']),  
    # ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
    ('SVM', SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)),
    ('nb', GaussianNB(var_smoothing= 1e-10)),
]
# ====== 3) Meta-learner ======
# meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    penalty='elasticnet',   # mix between L1 and L2
    l1_ratio=0.6,           # 0.0 -> pure L2, 1.0 -> pure L1
    C=1.0,                  # regularization strength
    class_weight='balanced',
    random_state=42
)
# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  # use probabilities from base models
    n_jobs=-1
)

# ====== 5) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='minority',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 6) Full pipeline with SVMSMOTE + Stacking ======
stacking_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote1', svmsmote),
    ('classifier', stacking_clf)
])

# ====== 7) Train and evaluate ======
print("\n🚀 Training Ensemble Stacking Model with SVMSMOTE...")
stacking_pipeline.fit(X_train_raw, y_train)

y_pred = stacking_pipeline.predict(X_test_raw)
y_proba = stacking_pipeline.predict_proba(X_test_raw)

# Evaluate
if len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba[:, 1])
else:
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
accuracy = accuracy_score(y_test, y_pred)
p_class, r_class, f1_class, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)
print(f1_w)
print("\n📊 Per-class metrics:")
for i in range(len(p_class)):
    print(f"Class {i}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")

print("\n✅ Ensemble Stacking Results")
print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")



🚀 Training Ensemble Stacking Model with SVMSMOTE...


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.7133989764417717

📊 Per-class metrics:
Class 0: P=0.8029, R=0.7679, F1=0.7850, Support=3193
Class 1: P=0.5298, R=0.5811, F1=0.5543, Support=1437

✅ Ensemble Stacking Results
Accuracy: 0.7099, F1: 0.6696, AUC: 0.7361


In [12]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# ========== 0) Assume dataset split ==========
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col != 'label']

# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)
dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}

# ====== 2) Base classifiers for stacking ======
# base_learners = [
#     ('lightgbm', LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
#     ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
#     ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42)),
# ]
from sklearn.ensemble import GradientBoostingClassifier

gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}
# 🔑 Notes on each setup:

# Balanced → vanilla RF, safe baseline.

# ShallowFast → small forest, faster training, may underfit.

# Deep → more complex trees, higher variance, may capture interactions.

# Regularized → controls overfitting with splits/leaves.

# Conservative → very large forest, but heavily regularized.

# RobustSubsample → adds randomness via subsampling + limited features.

# Lightweight → good for quick prototyping on large datasets.

# Heavy → huge ensemble, slow but highly stable (overkill for small datasets).

# 👉 Would you like me to also design a meta-list of “best RF defaults” (like your base_learners) so you can plug them into your stacking/blending pipeline directly?
rf_setups = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.6, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}

#Best: GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)
base_learners = [
    ('lightgbm', LGBMClassifier(
        n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    # ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
    #('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)),
    ('RandomForest', rf_setups['Regularized']),
    ('dt', dt_setups['Shallow']),  
    # ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
    ('SVM', SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)),
    ('nb', GaussianNB(var_smoothing= 1e-10)),
]
# ====== 3) Meta-learner ======
# meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    penalty='elasticnet',   # mix between L1 and L2
    l1_ratio=0.6,           # 0.0 -> pure L2, 1.0 -> pure L1
    C=1.0,                  # regularization strength
    class_weight='balanced',
    random_state=42
)
# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  # use probabilities from base models
    n_jobs=-1
)

# ====== 5) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='minority',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 6) Full pipeline with SVMSMOTE + Stacking ======
stacking_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote1', svmsmote),
    ('classifier', stacking_clf)
])

from sklearn.metrics import roc_auc_score

# ====== 7) Train and evaluate ======
print("\n🚀 Training Ensemble Stacking Model with SVMSMOTE...")
stacking_pipeline.fit(X_train_raw, y_train)

y_pred = stacking_pipeline.predict(X_test_raw)
y_proba = stacking_pipeline.predict_proba(X_test_raw)

# --- Global AUC ---
if len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba[:, 1])
else:
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

# --- AUC per class ---
auc_per_class = {}
classes = np.unique(y_test)
for i, cls in enumerate(classes):
    y_true_bin = (y_test == cls).astype(int)  # one-vs-rest
    auc_score = roc_auc_score(y_true_bin, y_proba[:, i])
    auc_per_class[cls] = auc_score

# Other metrics
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
accuracy = accuracy_score(y_test, y_pred)
p_class, r_class, f1_class, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)

print(f"\n📊 Per-class metrics:")
for i, cls in enumerate(classes):
    print(
        f"Class {cls}: "
        f"P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, "
        f"AUC={auc_per_class[cls]:.4f}, Support={support[i]}"
    )

print("\n✅ Ensemble Stacking Results")
print(f"Accuracy: {accuracy:.4f}, Macro-F1: {f1:.4f}, Weighted-F1: {f1_w:.4f}, Global AUC: {auc:.4f}")



🚀 Training Ensemble Stacking Model with SVMSMOTE...


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



📊 Per-class metrics:
Class 0.0: P=0.8029, R=0.7679, F1=0.7850, AUC=0.7360, Support=3193
Class 1.0: P=0.5298, R=0.5811, F1=0.5543, AUC=0.7360, Support=1437

✅ Ensemble Stacking Results
Accuracy: 0.7099, Macro-F1: 0.6696, Weighted-F1: 0.7134, Global AUC: 0.7360


In [27]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# ========== 0) Assume dataset split ==========
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col != 'label']

# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)
dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}

# ====== 2) Base classifiers for stacking ======
# base_learners = [
#     ('lightgbm', LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
#     ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
#     ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42)),
# ]
from sklearn.ensemble import GradientBoostingClassifier

gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}
# 🔑 Notes on each setup:

# Balanced → vanilla RF, safe baseline.

# ShallowFast → small forest, faster training, may underfit.

# Deep → more complex trees, higher variance, may capture interactions.

# Regularized → controls overfitting with splits/leaves.

# Conservative → very large forest, but heavily regularized.

# RobustSubsample → adds randomness via subsampling + limited features.

# Lightweight → good for quick prototyping on large datasets.

# Heavy → huge ensemble, slow but highly stable (overkill for small datasets).

# 👉 Would you like me to also design a meta-list of “best RF defaults” (like your base_learners) so you can plug them into your stacking/blending pipeline directly?
rf_setups = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.6, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}

#Best: GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)
# base_learners = [
#     ('lightgbm', LGBMClassifier(
#         n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
#     # ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
#     #('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)),
#     ('RandomForest', rf_setups['Regularized']),
#     ('dt', dt_setups['Shallow']),  
#     # ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
#     ('SVM', SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)),
#     ('nb', GaussianNB(var_smoothing= 1e-10)),
# ]
base_learners = [
    ('lightgbm', LGBMClassifier(
        n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    # ('xgboost', XGBClassifier(
    #     n_estimators=100, max_depth=6, learning_rate=0.06, subsample=0.8, colsample_bytree=0.8,
    #     random_state=42, eval_metric="logloss", use_label_encoder=False, verbosity=0)),
    ('rf', rf_setups['Regularized']),   # strong tree bagging
    ('nb', GaussianNB(var_smoothing= 1e-10)),
    # ('svm', SVC(kernel='rbf', C=10, gamma=0.01, probability=True, class_weight='balanced', random_state=42)),
    # ('lr', LogisticRegression(
    #     max_iter=2000, solver='saga', penalty='elasticnet',
    #     l1_ratio=0.5, C=0.5, class_weight='balanced', random_state=42))
]

# ====== 3) Meta-learner ======
# meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    penalty='elasticnet',   # mix between L1 and L2
    l1_ratio=0.6,           # 0.0 -> pure L2, 1.0 -> pure L1
    C=1.0,                  # regularization strength
    class_weight='balanced',
    random_state=42
)
# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  # use probabilities from base models
    n_jobs=-1
)

# ====== 5) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='minority',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 6) Full pipeline with SVMSMOTE + Stacking ======
stacking_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote1', svmsmote),
    ('classifier', stacking_clf)
])

# ====== 7) Train and evaluate ======
print("\n🚀 Training Ensemble Stacking Model with SVMSMOTE...")
stacking_pipeline.fit(X_train_raw, y_train)

y_pred = stacking_pipeline.predict(X_test_raw)
y_proba = stacking_pipeline.predict_proba(X_test_raw)

# Evaluate
if len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba[:, 1])
else:
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
accuracy = accuracy_score(y_test, y_pred)
p_class, r_class, f1_class, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)
print(f1_w)
print("\n📊 Per-class metrics:")
for i in range(len(p_class)):
    print(f"Class {i}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")

print("\n✅ Ensemble Stacking Results")
print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")



🚀 Training Ensemble Stacking Model with SVMSMOTE...


AttributeError: 'SVMSMOTE' object has no attribute '_validate_data'